In [1]:
import os
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")
from tqdm import tqdm 

# Decorter packages
from typing import List

# dataframe packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Tensorflow packages
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, 
                                     Reshape, 
                                    Input, 
                                    LSTM, 
                                    Dropout, 
                                    Conv1D, 
                                    MaxPooling1D,
                                    Conv2D, 
                                    MaxPooling2D, 
                                    GlobalAveragePooling1D, 
                                    Bidirectional,
                                    TimeDistributed,   
                                    Concatenate,
                                    Flatten
                                )
from tensorflow.keras.optimizers import (Adam, 
                                         AdamW)
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report

In [2]:
print("TensorFlow version:", tf.__version__)
print("Available physical devices:")
print(tf.config.list_physical_devices())

print("\nIs MPS available?")
print(tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.19.0
Available physical devices:
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Is MPS available?
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


#### Step -1 :  Load the Dataset

In [15]:
data = np.load("../eeg_processed/eeg_data.npz")
data["X"].shape, data["y"].shape


X = data["X"]
# Transpose to match (num_trails, n_samples, n_channels)
#X = np.transpose(X, (0, 2, 1))

y = data["y"]

print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

Shape of X: (1799, 16, 500)
Shape of y: (1799,)


#### Step -2 : Normalize Dataset

In [16]:
# Global Normalization
X = (X - np.mean(X)) / np.std(X)

print(f"Shape of X: {X.shape}")

Shape of X: (1799, 16, 500)


#### Step -3 : One Hot Encode labels (for classification)

In [17]:
y[y == 2] = 0 
y[y == 3] = 1
y[y == 4] = 2 
y[y == 5] = 3
y[y == 6] = 4 
y[y == 7] = 5

print(np.unique(y))

[0 1 2 3 4 5]


In [23]:
X = X[(y == 0) | (y == 1) | (y == 2)]
y = y[(y == 0) | (y == 1) | (y == 2)]

print(f"Shape of X : {(X.shape)}")
print(f"Shape of y : {(y.shape)}")

Shape of X : (899, 16, 500)
Shape of y : (899,)


In [24]:
np.unique_counts(y) # It's a balanced dataset

UniqueCountsResult(values=array([0, 1, 2]), counts=array([299, 300, 300]))

In [25]:
y = to_categorical(y, num_classes=len(np.unique(y)))

#### Step 4  Creating Tensorflow Dataset

In [26]:
data = tf.data.Dataset.from_tensor_slices((X, y))

2025-10-16 18:14:37.986174: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
2025-10-16 18:14:37.986336: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 18.00 GB
2025-10-16 18:14:37.986366: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 6.66 GB
I0000 00:00:1760656477.986433 2249777 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1760656477.986526 2249777 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [27]:
for x_batch, y_batch in data.take(1):
    print(x_batch.shape)
    print(y_batch.shape)

(16, 500)
(3,)


2025-10-16 18:14:39.082112: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


#### Step 5: Splitting the Tensorflow Dataset into Train, Test and Valid

In [28]:
train_split_ratio = 0.7
val_split_ratio = 0.15
test_split_ratio = 0.15

# Size of each split
train_size = int(train_split_ratio * X.shape[0])
val_size = int(val_split_ratio * X.shape[0])
test_size = int(test_split_ratio * X.shape[0])

# Shuffle the dataset first for a random split
data = data.shuffle(buffer_size= X.shape[0])

training_dataset = data.take(train_size)
val_dataset = data.take(val_size)
test_dataset = data.take(test_size)

print(f"Train dataset size: {len(list(training_dataset.as_numpy_iterator()))}")
print(f"Val dataset size: {len(list(val_dataset.as_numpy_iterator()))}")
print(f"Test dataset size: {len(list(test_dataset.as_numpy_iterator()))}")

Train dataset size: 629
Val dataset size: 134
Test dataset size: 134


2025-10-16 18:14:40.909331: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-10-16 18:14:40.932353: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [29]:
training_dataset = training_dataset.batch(32) # Training
val_dataset = val_dataset.batch(32) # Validation
test_dataset = test_dataset.batch(32) # Testing

#### Step 6: Conv and LSTM Model Architecture

In [33]:
class EEGCNNLSTM(Model):
    def __init__(self, num_classes):
        super(EEGCNNLSTM, self).__init__()

        # --- Reshape ---
        self.reshape_layer = Reshape((16, 500, 1))        

        # --- CNN feature extractor ---
        self.conv1 = Conv2D(16, (3, 3), activation='relu', padding='same')
        self.pool1 = MaxPooling2D((2, 2))

        self.conv2 = Conv2D(32, (5, 5), activation='relu', padding='same')
        self.pool2 = MaxPooling2D((2, 2))

        self.conv3 = Conv2D(64, (7, 7), activation='relu', padding='same')
        self.pool3 = MaxPooling2D((2, 2))

        self.dropout = Dropout(0.5)

        # Apply TimeDistributed 
        self.flatten = TimeDistributed(Flatten())
        self.fc_time = TimeDistributed(Dense(64, activation='relu'))

        # --- Temporal modeling ---
        self.lstm = LSTM(64, return_sequences=True)
        self.global_avg = GlobalAveragePooling1D()

        # --- Final layers ---
        self.concat = Concatenate()
        self.fc_final = Dense(64, activation='relu')
        self.out_layer = Dense(num_classes, activation='softmax')

    def call(self, inputs, training=False):

        # Reshape the input to accomdate the Convolution layers
        x = self.reshape_layer(inputs)

        # CNN feature extraction
        x = self.conv1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.pool3(x)

        if training:
            x = self.dropout(x)

        x = self.flatten(x)
        x = self.fc_time(x)

        # Expand for temporal modeling
        #x = tf.expand_dims(x, axis=1)  # (batch, time=1, features)
        lstm_out = self.lstm(x)

        # Global Average Pooling
        gap_out = self.global_avg(inputs)

        # Concatenate (to mimic diagram)
        concat_out = self.concat([gap_out, tf.reduce_mean(lstm_out, axis=1)])

        # Final classification
        dense_out = self.fc_final(concat_out)
        output = self.out_layer(dense_out)

        return output


# === Example usage ===
n_samples = 500     # number of time samples
n_channels = 16     # number of EEG electrodes
num_classes = 3     # binary classification

# Instantiate model
model = EEGCNNLSTM(num_classes)

# Build model (needed to show summary)
model.build(input_shape=(None, n_samples, n_channels, 1))
model.summary()


Model: "eegcnnlstm_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape_1 (Reshape)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ concatenate_1 (Concatenate)     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [34]:
f1_metric = tf.keras.metrics.F1Score(average='macro')

# Compile the Model
model.compile(optimizer= Adam(), 
                        loss = "categorical_crossentropy", 
                        metrics = ["accuracy", f1_metric])

In [35]:
history = model.fit(
    training_dataset,
    epochs = 100, 
    batch_size = 32, 
    validation_data = val_dataset
)

Epoch 1/100


2025-10-16 18:15:00.209849: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 88ms/step - accuracy: 0.3453 - f1_score: 0.3339 - loss: 1.1623 - val_accuracy: 0.4254 - val_f1_score: 0.4127 - val_loss: 1.0633
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.4409 - f1_score: 0.4015 - loss: 1.0645 - val_accuracy: 0.5149 - val_f1_score: 0.4823 - val_loss: 0.9749
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.4926 - f1_score: 0.4779 - loss: 1.0038 - val_accuracy: 0.5522 - val_f1_score: 0.5546 - val_loss: 0.9726
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.5283 - f1_score: 0.5244 - loss: 0.9686 - val_accuracy: 0.6119 - val_f1_score: 0.6123 - val_loss: 0.9108
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.5298 - f1_score: 0.5258 - loss: 0.9574 - val_accuracy: 0.5970 - val_f1_score: 0.5942 - val_loss: 0.9349
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.6049 - f1_score: 0.6042 - loss: 0.9063 - val_accuracy: 0.6567 - val_f1_score: 0.6509 - val_loss:

In [36]:
model.summary()

Model: "eegcnnlstm_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape_1 (Reshape)             │ (None, 16, 500, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 500, 16)    │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 8, 250, 32)     │        12,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 4, 125, 64)     │       100,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 2, 3968)        │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 2, 64)          │       254,016 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 2, 64)          │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ concatenate_1 (Concatenate)     │ (None, 564)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │        36,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,310,411 (5.00 MB)

 Trainable params: 436,803 (1.67 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 873,608 (3.33 MB)

In [45]:
# Validation Predictions
y_val_pred = model.predict(val_dataset)
y_val_pred = np.argmax(y_val_pred, axis = 1)

# Original Predictions
y_val_original = np.concatenate([y.numpy() for x, y in val_dataset])
y_val_original = np.argmax(y_val_original, axis = 1)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step


In [42]:
y_val_pred = model.predict(val_dataset)

1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step


In [46]:
print(classification_report(y_val_original, y_val_pred))

              precision    recall  f1-score   support

           0       0.41      0.36      0.39        47
           1       0.34      0.33      0.34        42
           2       0.33      0.38      0.35        45

    accuracy                           0.36       134
   macro avg       0.36      0.36      0.36       134
weighted avg       0.36      0.36      0.36       134



In [36]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_val_original, y_val_pred)

array([[ 6,  9, 11,  2,  5, 12],
       [10, 17,  5,  6,  6,  7],
       [ 6, 16,  5,  6,  5,  6],
       [ 8,  4,  8, 10,  6,  8],
       [ 9, 13,  9,  2,  9,  7],
       [ 4,  6,  8,  6,  3,  9]])